# AR-CBT V8 — numerical audit only
No human validation, no LLM calls, no new scores. Upload the complete V8 ZIP. Original model execution is not reproduced. The notebook is new analysis code, not an original experiment notebook.


In [ ]:
from pathlib import Path
import zipfile, hashlib, sys, subprocess
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one complete V8 ZIP archive.')
archive = Path(archives[0]); archive.write_bytes(uploaded[archives[0]])
base = Path('/content/AR_CBT_V8_AUDIT').resolve(); base.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    if sum(info.file_size for info in z.infolist()) > 150_000_000:
        raise ValueError('Archive unexpectedly large.')
    for info in z.infolist():
        target=(base/info.filename).resolve()
        if not target.is_relative_to(base):
            raise ValueError('Unsafe archive path.')
    z.extractall(base)
programs=list(base.rglob('reanalyse_scores.py'))
if len(programs)!=1:
    raise ValueError('Expected exactly one reanalyse_scores.py in the complete package.')
program=programs[0]
repro=program.parent.parent
package=repro.parent
manifest=package/'MANIFEST_SHA256.txt'
if not manifest.is_file(): raise FileNotFoundError('Manifest missing.')
for line in manifest.read_text().splitlines():
    if not line.strip(): continue
    digest,rel=line.split('  ',1)
    p=(package/rel).resolve()
    if not p.is_relative_to(package.resolve()): raise ValueError('Unsafe manifest path.')
    if hashlib.sha256(p.read_bytes()).hexdigest()!=digest:
        raise ValueError('Checksum mismatch: '+rel)
print('Package checksums verified. This verifies file integrity, not historical model provenance.')


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(program.parent/'requirements.txt')])
subprocess.check_call([sys.executable, str(program)])
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', str(program.parent), '-p', 'test_*.py', '-v'])


In [ ]:
import json
from IPython.display import display, Image
result=json.loads((repro/'results'/'analysis_results.json').read_text())
print(json.dumps(result,indent=2))
for name in ['Figure_1_Stored_Score_Association.png','Figure_2_Stored_Score_Discrepancies.png']:
    display(Image(filename=str(repro/'results'/name),width=800))


The figures and correlations describe stored automated-score series. Do not label them human–machine agreement or independent model validation. Recover original transcript/output/metadata links before stronger empirical interpretation.
